In [1]:
import os

In [2]:
%pwd

'c:\\Users\\DELL\\OneDrive\\Desktop\\Let us build\\Linear_regression_01\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\DELL\\OneDrive\\Desktop\\Let us build\\Linear_regression_01'

In [5]:
import box
print(box.__version__)

7.4.1


In [6]:
# 3 entites update
# from config.ymal
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataingestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path

In [8]:
from Linear_regression_01.constant import *
from Linear_regression_01.utils.common import read_yaml, create_directories

In [9]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataingestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataingestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file
        )

        return data_ingestion_config

In [10]:
import os
import urllib.request as request
import zipfile
from Linear_regression_01.logging import logger
from Linear_regression_01.utils.common import get_size

In [11]:
# 5 Conponents

class DataIngestion:
    def __init__(self, config: DataingestionConfig):
        self.config = config


    def download_files(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} Downloaded with following info \n{headers}")
        else:
            logger.info(f"file already exist of size: {get_size(Path(self.config.local_data_file))}")

In [12]:
# 6 Pipeline

try:
    config = configurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_files()

except Exception as e:
    raise e

[2026-06-07 21:26:04,958: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-07 21:26:04,975: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-07 21:26:04,979: INFO: common: created directory at artifacts]
[2026-06-07 21:26:04,983: INFO: common: created directory at artifacts/data_ingestion]
[2026-06-07 21:26:05,748: INFO: 3866272324: artifacts/data_ingestion/data.tar.gz Downloaded with following info 
Connection: close
Content-Length: 409488
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/octet-stream
ETag: "4e1a1e6699def5a17d41fed770852c4b91034b3a1f3989315c4ce6f238533173"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 9EB4:350F6D:45EF62:8DF719:6A25D3DC
Accept-Ranges: bytes
Date: Sun, 07 Jun 2026 20:26:05 GMT
Via: 1.1 varnish
X-Served-By: cache-lon4282-LON
X-

In [ ]:
import tarfile

class CleanDataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_files(self):
        if not os.path.exists(self.config.local_data_file):
            print(f"Starting download from {self.config.source_URL}...")
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            print(f"Downloaded successfully: {filename}")
        else:
            print("File already exists!")

    def extract_zip_file(self):
        """
        Extracts the tgz file into the unzip directory
        """
        # Create the unzip directory path from our config manager
        unzip_path = "artifacts/data_ingestion"
        os.makedirs(unzip_path, exist_ok=True)
        
        print(f"Extracting {self.config.local_data_file} to {unzip_path}...")
        with tarfile.open(self.config.local_data_file, "r:gz") as tar_ref:
            tar_ref.extractall(path=unzip_path)
        print("Extraction complete!")

In [19]:
try:
    print("--- STARTING FULL PIPELINE RUN ---")
    config_manager = configurationManager()
    ingestion_config = config_manager.get_data_ingestion_config()
    
    pipeline_component = CleanDataIngestion(config=ingestion_config)
    
    # 1. This will say "File already exists!" since we just downloaded it
    pipeline_component.download_files()
    
    # 2. This will unpack your dataset
    pipeline_component.extract_zip_file()
    
    print("--- ALL SANDBOX STEPS COMPLETED SUCCESSFULLY ---")

except Exception as e:
    raise e

--- STARTING FULL PIPELINE RUN ---
[2026-06-07 21:30:53,489: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-07 21:30:53,506: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-07 21:30:53,516: INFO: common: created directory at artifacts]
[2026-06-07 21:30:53,520: INFO: common: created directory at artifacts/data_ingestion]


File already exists!
Extracting artifacts/data_ingestion/data.tar.gz to artifacts/data_ingestion...
Extraction complete!
--- ALL SANDBOX STEPS COMPLETED SUCCESSFULLY ---
